In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("Data")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("Training shape:", train.shape)
print("Testing shape:", test.shape)

train.head()

Training shape: (9546, 131)
Testing shape: (4364, 130)


,Id,batch,label,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,...,feature_119,feature_120,feature_121,feature_122,feature_123,feature_124,feature_125,feature_126,feature_127,feature_128
0,11813,7,6,18426.1433,8.012311,4.637280,6.854782,10.755495,-2.700655,-3.956627,...,-7.649273,-15.691021,20123.4307,4.265258,6.735639,11.602448,16.781786,-4.716063,-7.207139,-15.563540
1,875,7,4,12881.6084,4.999426,2.700797,3.776042,7.609812,-1.719600,-2.495682,...,-8.172268,-13.557608,17969.0107,4.448080,5.905540,10.604961,13.701524,-4.375042,-7.002332,-12.966454
2,428,3,3,21118.7700,5.584744,5.340035,7.677719,8.328934,-3.701702,-4.710054,...,-7.272255,-9.995701,15885.3477,2.842874,5.417050,8.220536,9.267007,-3.889974,-6.030040,-8.723799
3,3249,7,2,19086.3298,7.184287,4.100390,6.309003,10.159742,-3.291821,-4.671168,...,-17.518615,-34.194218,37958.7178,12.322653,19.864813,51.314521,63.511047,-9.081936,-14.357400,-24.870645
4,2146,6,6,54701.7856,7.965081,16.761078,27.019008,36.757151,-9.944200,-13.916737,...,-5.870580,-11.506023,16154.8311,4.488141,4.503564,7.284964,10.996612,-3.259098,-5.218725,-11.085125


In [3]:
# Select the 128 sensor features
features = [f"feature_{i}" for i in range(1, 129)]

# Train on historical batches 1–6
train_data = train[train["batch"] <= 6]

# Validate on newer batch 7
val_data = train[train["batch"] == 7]

# Separate features and labels
X_train = train_data[features]
y_train = train_data["label"]

X_val = val_data[features]
y_val = val_data["label"]

# Check dimensions
print("Training:", X_train.shape)
print("Validation:", X_val.shape)

# Check gas class distribution
print("\nTraining class distribution:")
print(y_train.value_counts().sort_index())

print("\nValidation class distribution:")
print(y_val.value_counts().sort_index())

Training: (5933, 128)
Validation: (3613, 128)

Training class distribution:
label
1    1225
2    1579
3     541
4     484
5    1558
6     546
Name: count, dtype: int64

Validation class distribution:
label
1    649
2    662
3    360
4    744
5    630
6    568
Name: count, dtype: int64


In [4]:
from sklearn.preprocessing import StandardScaler

# Initialize scaler
scaler = StandardScaler()

# Learn mean and std from training data only
X_train_scaled = scaler.fit_transform(X_train)

# Apply same transformation to validation data
X_val_scaled = scaler.transform(X_val)

print("Before scaling:")
print(X_train.iloc[:3, :5])

print("\nAfter scaling:")
print(X_train_scaled[:3, :5])

Before scaling:
     feature_1  feature_2  feature_3  feature_4  feature_5
2   21118.7700   5.584744   5.340035   7.677719   8.328934
4   54701.7856   7.965081  16.761078  27.019008  36.757151
8  114887.2656  28.608835  34.667448  52.581881  56.304126

After scaling:
[[-0.64830901 -0.13467162 -0.6699523  -0.69788011 -0.70644238]
 [-0.25863971 -0.01075303 -0.13763397 -0.04636787  0.01285317]
 [ 0.43970241  1.06394554  0.69695609  0.81471873  0.50743403]]


In [5]:
from sklearn.svm import SVC
from sklearn.metrics import f1_score

# Create RBF-SVM model
svm_model = SVC(
    kernel="rbf",
    C=10,
    probability=True,
    random_state=42
)

# Train using standardized features
svm_model.fit(X_train_scaled, y_train)

# Predict on validation batch 7
svm_pred = svm_model.predict(X_val_scaled)

# Evaluate Macro-F1
svm_f1 = f1_score(y_val, svm_pred, average="macro")

print("SVM Validation Macro-F1:", round(svm_f1, 4))

C:\Users\rajab\202618066_Abhikarsh_Raj_DS605\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


SVM Validation Macro-F1: 0.878


In [6]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score

# Create Extra Trees model
et_model = ExtraTreesClassifier(
    n_estimators=220,
    max_features=0.7,
    random_state=42,
    n_jobs=-1
)

# Train on original features (no scaling required)
et_model.fit(X_train, y_train)

# Predict on validation batch 7
et_pred = et_model.predict(X_val)

# Calculate Macro-F1
et_f1 = f1_score(y_val, et_pred, average="macro")

print("Extra Trees Validation Macro-F1:", round(et_f1, 4))

Extra Trees Validation Macro-F1: 0.7919


In [7]:
import numpy as np
from sklearn.metrics import f1_score

# Get probabilities from both models
svm_prob = svm_model.predict_proba(X_val_scaled)
et_prob = et_model.predict_proba(X_val)

# Combine probabilities using weighted voting
ensemble_prob = 0.7 * svm_prob + 0.3 * et_prob

# Select class with highest combined probability
ensemble_pred = svm_model.classes_[
    np.argmax(ensemble_prob, axis=1)
]

# Calculate Macro-F1
ensemble_f1 = f1_score(
    y_val, ensemble_pred, average="macro"
)

print("SVM Macro-F1:", round(svm_f1, 4))
print("Extra Trees Macro-F1:", round(et_f1, 4))
print("Ensemble Macro-F1:", round(ensemble_f1, 4))

SVM Macro-F1: 0.878
Extra Trees Macro-F1: 0.7919
Ensemble Macro-F1: 0.8839


In [8]:
# Standardize full training and test data
X_full_scaled = scaler.fit_transform(train[features])
X_test_scaled = scaler.transform(test[features])

# Retrain both models using all training samples
svm_model.fit(X_full_scaled, train["label"])
et_model.fit(train[features], train["label"])

# Predict probabilities for test data
svm_test_prob = svm_model.predict_proba(X_test_scaled)
et_test_prob = et_model.predict_proba(test[features])

# Weighted soft voting
assert np.array_equal(svm_model.classes_, et_model.classes_)

final_prob = 0.7 * svm_test_prob + 0.3 * et_test_prob

final_pred = svm_model.classes_[np.argmax(final_prob, axis=1)]

# Create submission in required Id order
prediction_map = pd.Series(final_pred, index=test["Id"])

submission = sample[["Id"]].copy()
submission["label"] = submission["Id"].map(prediction_map).astype(int)

# Save CSV
submission.to_csv("submission.csv", index=False)

print("Submission shape:", submission.shape)
print(submission.head())

C:\Users\rajab\202618066_Abhikarsh_Raj_DS605\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Submission shape: (4364, 2)
     Id  label
0  2555      4
1  6408      5
2  2345      4
3   347      6
4  7030      6


In [9]:
# Check submission format
print("Missing values:")
print(submission.isnull().sum())

print("\nDuplicate IDs:", submission["Id"].duplicated().sum())

print("\nValid labels:",
      submission["label"].between(1, 6).all())

print("\nAll test IDs included:",
      set(submission["Id"]) == set(test["Id"]))

# Distribution of predicted gases
print("\nPredicted gas distribution:")
print(submission["label"].value_counts().sort_index())

# Distribution across future batches
print("\nPredictions by batch:")
print(pd.crosstab(
    test["batch"],
    final_pred,
    normalize="index"
).round(3))

Missing values:
Id       0
label    0
dtype: int64

Duplicate IDs: 0

Valid labels: True

All test IDs included: True

Predicted gas distribution:
label
1    969
2    641
3    612
4    722
5    540
6    880
Name: count, dtype: int64

Predictions by batch:
col_0      1      2      3      4      5      6
batch                                          
8      0.112  0.085  0.092  0.116  0.541  0.054
9      0.140  0.117  0.217  0.370  0.155  0.000
10     0.242  0.156  0.134  0.143  0.086  0.240


In [10]:
from sklearn.metrics import classification_report, confusion_matrix

# Evaluate individual gas classes
print("Ensemble Classification Report:")
print(classification_report(y_val, ensemble_pred))

# Check which gases are confused with each other
cm = confusion_matrix(y_val, ensemble_pred)

print("\nConfusion Matrix:")
print(cm)

Ensemble Classification Report:
              precision    recall  f1-score   support

           1       0.97      1.00      0.99       649
           2       0.76      1.00      0.86       662
           3       0.97      0.89      0.93       360
           4       1.00      0.57      0.73       744
           5       0.77      1.00      0.87       630
           6       1.00      0.87      0.93       568

    accuracy                           0.88      3613
   macro avg       0.91      0.89      0.88      3613
weighted avg       0.91      0.88      0.88      3613


Confusion Matrix:
[[649   0   0   0   0   0]
 [  0 662   0   0   0   0]
 [  0  34 321   0   5   0]
 [ 18 141  11 426 146   2]
 [  0   1   0   0 629   0]
 [  0  35   0   1  37 495]]


In [11]:
from scipy.stats import ks_2samp
import pandas as pd

# Reference: most recent labeled batch
reference = train[train["batch"] == 7]

for batch in [8, 9, 10]:
    future = test[test["batch"] == batch]

    drift = pd.Series({
        col: ks_2samp(reference[col], future[col]).statistic
        for col in features
    })

    print(f"\nBatch 7 vs {batch}")
    print("Median KS:", round(drift.median(), 3))
    print("Features with KS > 0.3:", (drift > 0.3).sum())
    print("Most shifted features:")
    print(drift.nlargest(5).round(3))


Batch 7 vs 8
Median KS: 0.423
Features with KS > 0.3: 98
Most shifted features:
feature_7    0.663
feature_6    0.657
feature_8    0.625
feature_1    0.610
feature_3    0.608
dtype: float64

Batch 7 vs 9
Median KS: 0.426
Features with KS > 0.3: 110
Most shifted features:
feature_95     0.648
feature_110    0.641
feature_94     0.634
feature_96     0.631
feature_111    0.622
dtype: float64

Batch 7 vs 10
Median KS: 0.198
Features with KS > 0.3: 8
Most shifted features:
feature_48    0.451
feature_64    0.418
feature_24    0.408
feature_32    0.405
feature_56    0.391
dtype: float64


In [12]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score
import numpy as np

for batch in [6, 7]:
    past = train[train["batch"] < batch]
    valid = train[train["batch"] == batch]

    # Fit scaler only on historical data
    sc = StandardScaler()
    X_tr = sc.fit_transform(past[features])
    X_v = sc.transform(valid[features])

    svm = SVC(C=10, kernel="rbf", probability=True,
              random_state=42)
    trees = ExtraTreesClassifier(
        n_estimators=220, max_features=0.7,
        random_state=42, n_jobs=-1
    )

    svm.fit(X_tr, past["label"])
    trees.fit(past[features], past["label"])

    p1 = svm.predict_proba(X_v)
    p2 = trees.predict_proba(valid[features])

    print(f"\nValidation Batch {batch}")

    for weight in [0.7, 0.9]:
        probabilities = weight * p1 + (1-weight) * p2
        predictions = svm.classes_[
            np.argmax(probabilities, axis=1)
        ]

        score = f1_score(valid["label"], predictions,
                         average="macro")

        print(f"SVM weight {weight}: F1 = {score:.4f}")

C:\Users\rajab\202618066_Abhikarsh_Raj_DS605\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



Validation Batch 6
SVM weight 0.7: F1 = 0.6406
SVM weight 0.9: F1 = 0.6385


C:\Users\rajab\202618066_Abhikarsh_Raj_DS605\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



Validation Batch 7
SVM weight 0.7: F1 = 0.8839
SVM weight 0.9: F1 = 0.8882


In [13]:
import numpy as np
from sklearn.pipeline import make_pipeline

def sensor_relative(X):
    # 16 sensors, 8 features per sensor
    x = np.asarray(X).reshape(-1, 16, 8)

    # Normalize across the 16 sensors
    mean = x.mean(axis=1, keepdims=True)
    std = x.std(axis=1, keepdims=True)

    return ((x - mean) / (std + 1e-6)).reshape(-1, 128)


for batch in [6, 7]:
    past = train[train["batch"] < batch]
    valid = train[train["batch"] == batch]

    X_tr = sensor_relative(past[features])
    X_v = sensor_relative(valid[features])

    model = make_pipeline(
        StandardScaler(),
        SVC(C=3, kernel="rbf")
    )

    model.fit(X_tr, past["label"])
    pred = model.predict(X_v)

    print(f"Batch {batch} Macro-F1:",
          round(f1_score(valid["label"], pred,
                         average="macro"), 4))

Batch 6 Macro-F1: 0.7135
Batch 7 Macro-F1: 0.9428


In [14]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from scipy.stats import mode
from sklearn.metrics import f1_score
import numpy as np

# Apply sensor-relative transformation
X_tr_rel = sensor_relative(X_train)
X_val_rel = sensor_relative(X_val)

models = []

# Train 3 independent SVM models
for c in [3, 5, 10]:
    model = make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", C=c)
    )

    model.fit(X_tr_rel, y_train)
    models.append(model)

# Get predictions from each model
predictions = np.array([
    model.predict(X_val_rel)
    for model in models
])

# Majority voting
ensemble_pred_rel = mode(
    predictions, axis=0, keepdims=False
).mode

print("Ensemble Macro-F1:",
      round(f1_score(y_val, ensemble_pred_rel,
                     average="macro"), 4))

Ensemble Macro-F1: 0.9437


In [15]:
# Transform complete training and test data
X_full_rel = sensor_relative(train[features])
X_test_rel = sensor_relative(test[features])

# Retrain all three SVM models on batches 1–7
for model in models:
    model.fit(X_full_rel, train["label"])

# Predictions from each model
test_predictions = np.array([
    model.predict(X_test_rel)
    for model in models
])

# Majority voting
final_pred_rel = mode(
    test_predictions, axis=0, keepdims=False
).mode

# Create submission using correct IDs
prediction_map = pd.Series(final_pred_rel, index=test["Id"])

submission_rel = sample[["Id"]].copy()
submission_rel["label"] = submission_rel["Id"].map(prediction_map).astype(int)

# Save without overwriting previous submission
submission_rel.to_csv("submission_relative.csv", index=False)

print("Submission shape:", submission_rel.shape)
print("\nPredicted class distribution:")
print(submission_rel["label"].value_counts().sort_index())

print("\nPredictions by batch:")
print(pd.crosstab(
    test["batch"],
    final_pred_rel,
    normalize="index"
).round(3))

Submission shape: (4364, 2)

Predicted class distribution:
label
1    1570
2     715
3     633
4     275
5     617
6     554
Name: count, dtype: int64

Predictions by batch:
col_0      1      2      3      4      5      6
batch                                          
8      0.109  0.105  0.102  0.109  0.517  0.058
9      0.272  0.128  0.213  0.066  0.147  0.174
10     0.392  0.173  0.140  0.059  0.110  0.126


In [16]:
from sklearn.metrics import classification_report

print("Original Ensemble:")
print(classification_report(y_val, ensemble_pred))

print("\nSensor-Relative Ensemble:")
print(classification_report(y_val, ensemble_pred_rel))

Original Ensemble:
              precision    recall  f1-score   support

           1       0.97      1.00      0.99       649
           2       0.76      1.00      0.86       662
           3       0.97      0.89      0.93       360
           4       1.00      0.57      0.73       744
           5       0.77      1.00      0.87       630
           6       1.00      0.87      0.93       568

    accuracy                           0.88      3613
   macro avg       0.91      0.89      0.88      3613
weighted avg       0.91      0.88      0.88      3613


Sensor-Relative Ensemble:
              precision    recall  f1-score   support

           1       0.79      1.00      0.88       649
           2       0.96      1.00      0.98       662
           3       1.00      0.93      0.96       360
           4       0.99      0.75      0.85       744
           5       0.98      1.00      0.99       630
           6       1.00      1.00      1.00       568

    accuracy                   

In [17]:
comparison = test[["Id", "batch"]].copy()

comparison["original"] = comparison["Id"].map(
    submission.set_index("Id")["label"]
)

comparison["relative"] = comparison["Id"].map(
    submission_rel.set_index("Id")["label"]
)

comparison["different"] = (
    comparison["original"] != comparison["relative"]
)

print("Prediction disagreement by batch:")
print(
    comparison.groupby("batch")["different"]
    .mean()
    .mul(100)
    .round(2)
)

Prediction disagreement by batch:
batch
8      6.46
9     34.47
10    39.03
Name: different, dtype: float64


In [18]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(
    y_val,
    ensemble_pred_rel,
    labels=[1, 2, 3, 4, 5, 6]
)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {i}" for i in range(1, 7)],
    columns=[f"Predicted {i}" for i in range(1, 7)]
)

display(cm_df)

,Predicted 1,Predicted 2,Predicted 3,Predicted 4,Predicted 5,Predicted 6
Actual 1,649,0,0,0,0,0
Actual 2,0,662,0,0,0,0
Actual 3,19,5,334,0,2,0
Actual 4,156,22,0,555,11,0
Actual 5,0,1,0,2,627,0
Actual 6,0,0,0,1,0,567


In [19]:
assert submission_rel.shape == (4364, 2)
assert list(submission_rel.columns) == ["Id", "label"]
assert submission_rel["Id"].is_unique
assert set(submission_rel["Id"]) == set(test["Id"])
assert submission_rel["label"].between(1, 6).all()
assert not submission_rel.isna().any().any()

submission_rel.to_csv("submission_relative.csv", index=False)

print("Submission ready!")

Submission ready!


In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import f1_score
import numpy as np

for batch in [6, 7]:

    tr = train[train["batch"] < batch]
    val = train[train["batch"] == batch]

    # 1. Sensor-relative SVM
    svm = make_pipeline(
        StandardScaler(),
        SVC(C=5, kernel="rbf")
    )

    svm.fit(sensor_relative(tr[features]), tr["label"])
    p_svm = svm.predict(sensor_relative(val[features]))

    # 2. Logistic Regression
    lr = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=0.1, max_iter=2000)
    )

    lr.fit(tr[features], tr["label"])
    p_lr = lr.predict(val[features])

    # 3. Random Forest
    rf = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(tr[features], tr["label"])
    p_rf = rf.predict(val[features])

    # Majority voting; SVM breaks three-way ties
    final = np.where(p_lr == p_rf, p_lr, p_svm)

    print(f"Batch {batch}:")
    print("SVM:", round(f1_score(val["label"], p_svm, average="macro"), 4))
    print("LR:", round(f1_score(val["label"], p_lr, average="macro"), 4))
    print("RF:", round(f1_score(val["label"], p_rf, average="macro"), 4))
    print("Hybrid:", round(f1_score(val["label"], final, average="macro"), 4))

Batch 6:
SVM: 0.7151
LR: 0.6698
RF: 0.626
Hybrid: 0.6428
Batch 7:
SVM: 0.9439
LR: 0.8001
RF: 0.7292
Hybrid: 0.9


In [21]:
from sklearn.calibration import CalibratedClassifierCV

for batch in [6, 7]:

    tr = train[train["batch"] < batch]
    val = train[train["batch"] == batch]

    # Model 1: Sensor-relative SVM
    svm = CalibratedClassifierCV(
        make_pipeline(
            StandardScaler(),
            SVC(C=5, kernel="rbf")
        ),
        method="sigmoid",
        cv=3,
        ensemble=False
    )

    # Model 2: Logistic Regression
    lr = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=0.1, max_iter=2000)
    )

    # Model 3: Random Forest
    rf = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

    # Train models
    svm.fit(sensor_relative(tr[features]), tr["label"])
    lr.fit(tr[features], tr["label"])
    rf.fit(tr[features], tr["label"])

    # Predict probabilities
    p1 = svm.predict_proba(sensor_relative(val[features]))
    p2 = lr.predict_proba(val[features])
    p3 = rf.predict_proba(val[features])

    assert np.array_equal(svm.classes_, lr.classes_)
    assert np.array_equal(svm.classes_, rf.classes_)

    # Weighted soft voting
    combined = 0.50*p1 + 0.33*p2 + 0.17*p3

    pred = svm.classes_[np.argmax(combined, axis=1)]

    score = f1_score(val["label"], pred, average="macro")

    print(f"Batch {batch} Weighted Ensemble F1: {score:.4f}")

Batch 6 Weighted Ensemble F1: 0.7151
Batch 7 Weighted Ensemble F1: 0.9428


In [22]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import f1_score
import numpy as np

def combined_features(X):
    original = X.to_numpy()
    relative = sensor_relative(X)

    return np.hstack([original, relative])


for batch in [6, 7]:

    tr = train[train["batch"] < batch]
    val = train[train["batch"] == batch]

    X_tr = combined_features(tr[features])
    X_val = combined_features(val[features])

    model = make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", C=5)
    )

    model.fit(X_tr, tr["label"])
    pred = model.predict(X_val)

    score = f1_score(
        val["label"], pred, average="macro"
    )

    print(f"Batch {batch} Macro-F1: {score:.4f}")

Batch 6 Macro-F1: 0.6583
Batch 7 Macro-F1: 0.9425


In [23]:
from sklearn.decomposition import PCA

for batch in [6, 7]:

    tr = train[train["batch"] < batch]
    val = train[train["batch"] == batch]

    # Sensor-relative features
    X_tr = sensor_relative(tr[features])
    X_v = sensor_relative(val[features])

    # Standardization + PCA + SVM
    model = make_pipeline(
        StandardScaler(),
        PCA(
            n_components=24,
            svd_solver="randomized",
            random_state=42
        ),
        SVC(kernel="rbf", C=5)
    )

    model.fit(X_tr, tr["label"])
    predictions = model.predict(X_v)

    print(
        f"Batch {batch} Macro-F1:",
        round(f1_score(val["label"], predictions,
                       average="macro"), 4)
    )

Batch 6 Macro-F1: 0.7403
Batch 7 Macro-F1: 0.9358


In [24]:
from sklearn.calibration import CalibratedClassifierCV

for batch in [6, 7]:
    tr = train[train["batch"] < batch]
    val = train[train["batch"] == batch]

    X_tr = sensor_relative(tr[features])
    X_val = sensor_relative(val[features])

    models_pca = []

    for n in [24, 48]:
        model = CalibratedClassifierCV(
            make_pipeline(
                StandardScaler(),
                PCA(n_components=n,
                    svd_solver="randomized",
                    random_state=42),
                SVC(kernel="rbf", C=5)
            ),
            method="sigmoid",
            cv=3,
            ensemble=False
        )

        model.fit(X_tr, tr["label"])
        models_pca.append(model)

    # 70% PCA-24 + 30% PCA-48
    probabilities = (
        0.7 * models_pca[0].predict_proba(X_val)
        + 0.3 * models_pca[1].predict_proba(X_val)
    )

    pred = models_pca[0].classes_[
        np.argmax(probabilities, axis=1)
    ]

    print(f"Batch {batch} Macro-F1:",
          round(f1_score(val["label"], pred,
                         average="macro"), 4))

Batch 6 Macro-F1: 0.7403
Batch 7 Macro-F1: 0.9352


In [25]:
# 1. Prepare full training and test data
X_full = sensor_relative(train[features])
X_test = sensor_relative(test[features])
y_full = train["label"]

# 2. Train PCA-24 and PCA-48 SVM models
final_models = []

for n in [24, 48]:
    model = CalibratedClassifierCV(
        make_pipeline(
            StandardScaler(),
            PCA(
                n_components=n,
                svd_solver="randomized",
                random_state=42
            ),
            SVC(kernel="rbf", C=5)
        ),
        method="sigmoid",
        cv=3,
        ensemble=False
    )

    model.fit(X_full, y_full)
    final_models.append(model)

# 3. Predict probabilities
p24 = final_models[0].predict_proba(X_test)
p48 = final_models[1].predict_proba(X_test)

# 4. Weighted soft voting (70% + 30%)
assert np.array_equal(
    final_models[0].classes_,
    final_models[1].classes_
)

combined = 0.7 * p24 + 0.3 * p48

final_predictions = final_models[0].classes_[
    np.argmax(combined, axis=1)
]

# 5. Create submission file
prediction_map = pd.Series(
    final_predictions,
    index=test["Id"]
)

submission_pca = sample[["Id"]].copy()
submission_pca["label"] = (
    submission_pca["Id"].map(prediction_map).astype(int)
)

# 6. Validate submission
assert submission_pca.shape == (4364, 2)
assert submission_pca["Id"].is_unique
assert set(submission_pca["Id"]) == set(test["Id"])
assert submission_pca["label"].between(1, 6).all()
assert not submission_pca.isna().any().any()

# 7. Save submission
submission_pca.to_csv("submission_pca.csv", index=False)

print("Submission ready!")
print("Shape:", submission_pca.shape)

print("\nPredicted class distribution:")
print(submission_pca["label"].value_counts().sort_index())

print("\nPredictions by batch:")
print(pd.crosstab(
    test["batch"],
    final_predictions,
    normalize="index"
).round(3))

Submission ready!
Shape: (4364, 2)

Predicted class distribution:
label
1    1553
2     688
3     702
4     269
5     565
6     587
Name: count, dtype: int64

Predictions by batch:
col_0      1      2      3      4      5      6
batch                                          
8      0.102  0.105  0.099  0.109  0.517  0.068
9      0.266  0.134  0.211  0.068  0.149  0.172
10     0.388  0.165  0.159  0.057  0.095  0.135
